In [1]:
#Import and install necessary packages
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from functools import reduce
import scipy.stats as stats
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [3]:
#Load data from file directory
mortality_socio_22_non_f = pd.read_csv("/content/drive/MyDrive/Portfolio/30 Days Challenge/Day 8/mortality_socio_22_non_f.csv")

In [4]:
#Setup data for modeling
predictors = ['gdp_per_capita', 'health_exp_pct_gdp', 'dtp3_immunization_pct', 'basic_sanitation_pct']
target = 'per_1000'

X = mortality_socio_22_non_f[predictors]
y = mortality_socio_22_non_f[target]

In [5]:
#Train-test split
# 80/20 split, random_state fixed for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
#Standardize (fit on train only, apply to both — avoids data leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
# Fit three models
models = {
    "OLS": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=1.0),
}

In [8]:
results = []

for name, mdl in models.items():
    mdl.fit(X_train_scaled, y_train)
    preds = mdl.predict(X_test_scaled)

    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)

    results.append({"model": name, "R2": r2, "RMSE": rmse, "MAE": mae})

    # Show which coefficients each model kept/shrank
    coefs = pd.Series(mdl.coef_, index=predictors)
    print(f"\n{name} coefficients:")
    print(coefs)

results_df = pd.DataFrame(results)
print("\n=== Model comparison on held-out test set ===")
print(results_df)


OLS coefficients:
gdp_per_capita           -1.931352
health_exp_pct_gdp       -0.903248
dtp3_immunization_pct    -3.333430
basic_sanitation_pct    -19.730877
dtype: float64

Ridge coefficients:
gdp_per_capita           -1.974081
health_exp_pct_gdp       -0.930524
dtp3_immunization_pct    -3.377428
basic_sanitation_pct    -19.540833
dtype: float64

Lasso coefficients:
gdp_per_capita           -1.318848
health_exp_pct_gdp       -0.139945
dtp3_immunization_pct    -2.836508
basic_sanitation_pct    -19.393555
dtype: float64

=== Model comparison on held-out test set ===
   model        R2       RMSE        MAE
0    OLS  0.326209  58.922719  16.904609
1  Ridge  0.325153  58.968872  16.931457
2  Lasso  0.315758  59.377910  17.506609


In [9]:
test_countries = mortality_socio_22_non_f.loc[X_test.index, 'entity']
print(test_countries.tolist())
print('Central African Republic' in test_countries.values)

['Singapore', 'Central African Republic', 'Panama', 'Canada', 'Spain', 'United Kingdom', 'Uzbekistan', 'Eswatini', 'Mozambique', 'Greece', 'Belgium', 'Turks and Caicos Islands', 'Senegal', 'Dominican Republic', 'Hungary', 'Lesotho', 'Bulgaria', 'Netherlands', 'Italy', 'Nepal', 'Bolivia', 'Belize', 'Chad', 'Bhutan', 'Bangladesh', 'Austria', 'Kuwait', 'Mauritius', 'Jamaica', 'France', 'Denmark', 'Gabon', 'Tajikistan', 'Nicaragua']
True


In [10]:
# Trying for excluded Central Africa Republic from data
df_no_car = mortality_socio_22_non_f[mortality_socio_22_non_f['entity'] != 'Central African Republic'].copy()

predictors = ['gdp_per_capita', 'health_exp_pct_gdp', 'dtp3_immunization_pct', 'basic_sanitation_pct']
target = 'per_1000'

from sklearn.preprocessing import StandardScaler
scaler_no_car = StandardScaler()
X_no_car_scaled = pd.DataFrame(
    scaler_no_car.fit_transform(df_no_car[predictors]),
    columns=predictors,
    index=df_no_car.index   # critical: keep the same index as df_no_car
)

In [11]:
import statsmodels.api as sm
X_no_car_scaled = sm.add_constant(X_no_car_scaled)
y_no_car = df_no_car[target]

In [12]:
 # exactly 4 columns, no constant
X_no_car = df_no_car[predictors].copy()
y_no_car = df_no_car[target].copy()

#Train test split for the data excluding Central republic African
X_train, X_test, y_train, y_test = train_test_split(X_no_car, y_no_car, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [13]:
# Fit the three models
models = {
    "OLS": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=1.0),
}


results = []

for name, mdl in models.items():
    mdl.fit(X_train_scaled, y_train)
    preds = mdl.predict(X_test_scaled)

    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)

    results.append({"model": name, "R2": r2, "RMSE": rmse, "MAE": mae})

    # Show which coefficients each model kept/shrank
    coefs = pd.Series(mdl.coef_, index=predictors)
    print(f"\n{name} coefficients:")
    print(coefs)

results_df = pd.DataFrame(results)
print("\n=== Model comparison on held-out test set ===")
print(results_df)


OLS coefficients:
gdp_per_capita           -2.169329
health_exp_pct_gdp       -0.990830
dtp3_immunization_pct    -2.822243
basic_sanitation_pct    -19.755739
dtype: float64

Ridge coefficients:
gdp_per_capita           -2.213067
health_exp_pct_gdp       -1.008018
dtp3_immunization_pct    -2.872962
basic_sanitation_pct    -19.563824
dtype: float64

Lasso coefficients:
gdp_per_capita           -1.559967
health_exp_pct_gdp       -0.183038
dtp3_immunization_pct    -2.308266
basic_sanitation_pct    -19.396113
dtype: float64

=== Model comparison on held-out test set ===
   model        R2      RMSE       MAE
0    OLS  0.819114  9.721270  6.905012
1  Ridge  0.819213  9.718603  6.936345
2  Lasso  0.817731  9.758350  7.372470


In [14]:
print(X_no_car_scaled.shape)          # should be (168, 4) — if it's (168, 5), that's the bug
print(X_no_car_scaled.columns.tolist())

(167, 5)
['const', 'gdp_per_capita', 'health_exp_pct_gdp', 'dtp3_immunization_pct', 'basic_sanitation_pct']
